# Bakalauro baigiamasis darbas: Skatinamojo mokymosi modelių tyrimas AAC sistemose

## 1. Importuotos bibliotekos

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict, deque
import random
from dataclasses import dataclass, field
from typing import Optional, Tuple, List, Dict, Any
from datetime import datetime
import copy
import itertools
import warnings
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 2. Konfigūracija ir parametrai

### Metrikos:
- Atitikties rodiklis (%) = (sėkmingos sesijos / visos sesijos) × 100
- Vidutinis bandymų skaičius = suma bandymų / sesijų skaičius

### Sesija:
- Sėkminga sesija = vaikas pasirinko norimą kortelę per ≤ MAX_ATTEMPTS bandymų
- Nesėkminga sesija = per MAX_ATTEMPTS bandymų vaikas nepasirinko norimos korteles

In [ ]:
N_RUNS = 3  # Kiek kartų pakartoti kiekvieną konfigūraciją
MAX_ATTEMPTS = 5  # Maksimalus bandymų skaičius vienoje sesijoje

PARAM_GRID = {
    'alpha': [0.3],           # Mokymosi greitis
    'gamma': [0.6],          # Nuolaidos faktorius
    'epsilon': [0.15],      # Tyrinėjimo tikimybė
    'exploration_method': ['epsilon_greedy', 'decaying_epsilon', 'ucb'],
    'strategy': ['A', 'B', 'C', 'D', 'E'],
    'similarity_method': ['cosine', 'euclidean'],
}

UCB_C = np.sqrt(2)

EPSILON_DECAY = 0.995
EPSILON_MIN = 0.01

MODEL_NAMES = ['Q-Learning', 'SARSA', 'Expected-SARSA', 'Double-Q-Learning', 'Actor-Critic']

CHILD_PROFILES = ['consistent', 'noisy']

REWARDS = {
    'success': 1.0,     # Sėkmė (bet kuriame bandyme)
    'rejection': -1.0,  # Atmetimas (pirma / pagrindinė kortelė)
    'other_card': -0.2  # Antroji kortelė – papildoma bauda
}

print(f"Modeliai: {MODEL_NAMES}")
print(f"Strategijos: {PARAM_GRID['strategy']}")
print(f"Tyrinėjimo metodai: {PARAM_GRID['exploration_method']}")
print(f"Vaiko profiliai: {CHILD_PROFILES}")
print(f"Max bandymų per sesiją: {MAX_ATTEMPTS}")
print(f"Pakartojimai: {N_RUNS}")

## 3. Produktų duomenų bazė

In [ ]:
FOOD_DATABASE = {
    'Obuolys': {'kietumas': 0.8, 'saldumas': 0.7, 'rugstingumas': 0.4, 'forma': 0.0, 'tekstura': 0.2, 'spalva': 0.0},
    'Žaliasis obuolys': {'kietumas': 0.8, 'saldumas': 0.5, 'rugstingumas': 0.7, 'forma': 0.0, 'tekstura': 0.2, 'spalva': 0.66},
    'Kriaušė': {'kietumas': 0.6, 'saldumas': 0.8, 'rugstingumas': 0.2, 'forma': 0.3, 'tekstura': 0.3, 'spalva': 0.5},
    'Bananas': {'kietumas': 0.3, 'saldumas': 0.9, 'rugstingumas': 0.1, 'forma': 1.0, 'tekstura': 0.0, 'spalva': 0.33},
    'Persikai': {'kietumas': 0.4, 'saldumas': 0.8, 'rugstingumas': 0.2, 'forma': 0.0, 'tekstura': 0.8, 'spalva': 0.15},
    'Mangas': {'kietumas': 0.5, 'saldumas': 0.9, 'rugstingumas': 0.3, 'forma': 0.2, 'tekstura': 0.1, 'spalva': 0.25},
    'Apelsinas': {'kietumas': 0.6, 'saldumas': 0.7, 'rugstingumas': 0.5, 'forma': 0.0, 'tekstura': 0.4, 'spalva': 0.2},
    'Mandarinas': {'kietumas': 0.5, 'saldumas': 0.8, 'rugstingumas': 0.4, 'forma': 0.0, 'tekstura': 0.3, 'spalva': 0.2},
    'Arbūzas': {'kietumas': 0.3, 'saldumas': 0.8, 'rugstingumas': 0.1, 'forma': 0.0, 'tekstura': 0.0, 'spalva': 0.0},
    'Melionas': {'kietumas': 0.4, 'saldumas': 0.7, 'rugstingumas': 0.1, 'forma': 0.0, 'tekstura': 0.1, 'spalva': 0.4},
    'Ananasas': {'kietumas': 0.6, 'saldumas': 0.8, 'rugstingumas': 0.6, 'forma': 0.5, 'tekstura': 0.9, 'spalva': 0.33},
    'Kiviai': {'kietumas': 0.5, 'saldumas': 0.6, 'rugstingumas': 0.5, 'forma': 0.0, 'tekstura': 0.6, 'spalva': 0.66},
    'Vyšnios': {'kietumas': 0.7, 'saldumas': 0.7, 'rugstingumas': 0.3, 'forma': 0.0, 'tekstura': 0.1, 'spalva': 0.0},
    'Braškės': {'kietumas': 0.4, 'saldumas': 0.8, 'rugstingumas': 0.2, 'forma': 0.4, 'tekstura': 0.5, 'spalva': 0.0},
    'Mėlynės': {'kietumas': 0.5, 'saldumas': 0.6, 'rugstingumas': 0.4, 'forma': 0.0, 'tekstura': 0.1, 'spalva': 1.0},
    'Avietės': {'kietumas': 0.3, 'saldumas': 0.7, 'rugstingumas': 0.3, 'forma': 0.1, 'tekstura': 0.7, 'spalva': 0.05},
    'Vynuogės': {'kietumas': 0.6, 'saldumas': 0.8, 'rugstingumas': 0.2, 'forma': 0.0, 'tekstura': 0.0, 'spalva': 0.8}
}

PROPERTY_COLS = ['kietumas', 'saldumas', 'rugstingumas', 'forma', 'tekstura', 'spalva']

food_data = [{'pavadinimas': name, **props} for name, props in FOOD_DATABASE.items()]
FOOD_DF = pd.DataFrame(food_data)
FOOD_LIST = FOOD_DF['pavadinimas'].tolist()

print(f"{len(FOOD_DF)} produktų, {len(PROPERTY_COLS)} savybės")

## 4. Panašumo matricos

In [4]:
def compute_similarity_matrix(method='cosine'):
    vectors = FOOD_DF[PROPERTY_COLS].values
    n = len(vectors)

    if method == 'cosine':
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1
        normalized = vectors / norms
        sim = normalized @ normalized.T
    else:  # euclidean
        sim = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i != j:
                    dist = np.sqrt(np.sum((vectors[i] - vectors[j])**2))
                    sim[i][j] = 1 / (1 + dist)

    np.fill_diagonal(sim, 0)
    return sim

SIMILARITY_MATRICES = {
    'cosine': compute_similarity_matrix('cosine'),
    'euclidean': compute_similarity_matrix('euclidean')
}

## 5. Būsena (State)

In [5]:
@dataclass(frozen=True)
class State:
    rejections: int
    last_action: str

    def __repr__(self):
        return f"S(rej={self.rejections},{self.last_action[:3]})"

## 6. Virtualaus vaiko norų sąrašas

In [ ]:
CHILD_WANTS = [
    "Bananas", "Bananas", "Mangas", "Bananas", "Mangas",
    "Bananas", "Bananas", "Mangas", "Bananas", "Mandarinas",
    "Bananas", "Bananas", "Mangas", "Bananas", "Mangas",
    "Bananas", "Bananas", "Obuolys", "Obuolys", "Kriaušė",
    "Kriaušė", "Bananas", "Obuolys", "Kriaušė", "Bananas",
    "Obuolys", "Kriaušė", "Mandarinas", "Obuolys", "Kriaušė",
    "Bananas", "Obuolys", "Obuolys", "Kriaušė", "Bananas",
    "Mangas", "Bananas", "Obuolys", "Kriaušė", "Mandarinas",
    "Bananas", "Bananas", "Mėlynės", "Mėlynės", "Vynuogės",
    "Vynuogės", "Bananas", "Mėlynės", "Vynuogės", "Mandarinas",
    "Mėlynės", "Vynuogės", "Bananas", "Obuolys", "Kriaušė",
    "Mėlynės", "Vynuogės", "Bananas", "Mandarinas", "Mėlynės",
    "Vynuogės", "Bananas", "Mangas", "Bananas", "Bananas",
    "Obuolys", "Bananas", "Mandarinas", "Bananas", "Kriaušė",
    "Bananas", "Mėlynės", "Bananas", "Vynuogės", "Bananas",
    "Bananas", "Mandarinas", "Bananas", "Mangas", "Bananas",
    "Obuolys", "Bananas", "Obuolys", "Bananas", "Bananas",
    "Mėlynės", "Mandarinas", "Obuolys", "Mandarinas", "Bananas",
    "Mandarinas", "Obuolys", "Mangas", "Mėlynės", "Bananas",
    "Vynuogės", "Mangas", "Mėlynės", "Obuolys", "Bananas"
]

print(f"Norų sąrašas: {len(CHILD_WANTS)} norų/sesijų")

## 7. Rezultatų logger'is

In [7]:
class ResultsLogger:
    def __init__(self):
        self.records = []
        self.session_results = []  # 1 = sėkmė, 0 = nesėkmė
        self.attempts_per_session = []
        self.cumulative_success = 0
        self.cumulative_rejection = 0
        self.max_success_streak = 0
        self._current_streak = 0

    def reset(self):
        self.records = []
        self.session_results = []
        self.attempts_per_session = []
        self.cumulative_success = 0
        self.cumulative_rejection = 0
        self.max_success_streak = 0
        self._current_streak = 0

    def log_attempt(self, **kwargs):
        result = kwargs.get('result', '')
        if result == 'SUCCESS':
            self.cumulative_success += 1
            self._current_streak += 1
            self.max_success_streak = max(self.max_success_streak, self._current_streak)
        else:
            self.cumulative_rejection += 1
            self._current_streak = 0
        self.records.append(kwargs)

    def log_session(self, success: bool, attempts: int):
        self.session_results.append(1 if success else 0)
        self.attempts_per_session.append(attempts)


    def get_success_rate(self) -> float:
        if not self.session_results:
            return 0.0
        return np.mean(self.session_results) * 100

    def get_avg_attempts(self) -> float:
        if not self.attempts_per_session:
            return 0.0
        return np.mean(self.attempts_per_session)


    def get_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)

    def save_csv(self, filename: str):
        self.get_dataframe().to_csv(filename, index=False)
        print(f"Išsaugota: {filename}")

## 8. Stochastinis virtualus vaikas

### Nuoseklus profilis (consistent)
- Visada renkasi kortelę pagal norų sąrašą



### Triukšmingas profilis (noisy)

Jei norima kortelė yra tarp rodomų:
- 80% renkasi tiksliai pagal norų sąrašą
- 10% motorinė klaida (netyčia paspaudė kitą)
- 10% dėmesio trūkumas (nepastebėjo norimos, atmeta norimą)

Jei norimos kortelės tarp rodomų nebuvo, yra 5% tikimybė, kad virtualus vaikas pasirinks bet kokią kortelę.

In [8]:
class VirtualChild:

    def __init__(self, wants_list: List[str], similarity_matrix=None):
        self.wants_list = wants_list
        self.current_want_index = 0
        self.similarity_matrix = similarity_matrix if similarity_matrix is not None else SIMILARITY_MATRICES['cosine']
        self.profile_name = "base"

    def get_current_want(self) -> Optional[str]:
        if self.current_want_index < len(self.wants_list):
            return self.wants_list[self.current_want_index]
        return None

    def next_want(self):
        self.current_want_index += 1

    def reset(self):
        self.current_want_index = 0

    def _get_similarity(self, card1: str, card2: str) -> float:
        idx1 = FOOD_LIST.index(card1)
        idx2 = FOOD_LIST.index(card2)
        return self.similarity_matrix[idx1][idx2]

    def react_to_cards(self, card1: str, card2: str) -> Optional[str]:
        raise NotImplementedError


class ConsistentChild(VirtualChild):

    def __init__(self, wants_list: List[str], similarity_matrix=None):
        super().__init__(wants_list, similarity_matrix)
        self.profile_name = "consistent"
        self.exact_prob = 0.95
        self.similar_prob = 0.05

    def react_to_cards(self, card1, card2):
        want = self.get_current_want()
        if want == card1: return card1
        if want == card2: return card2
        return None


class NoisyChild(VirtualChild):

    def __init__(self, wants_list: List[str], similarity_matrix=None):
        super().__init__(wants_list, similarity_matrix)
        self.profile_name = "noisy"
        self.correct_prob = 0.80
        self.motor_error_prob = 0.10
        self.attention_error_prob = 0.10

    def react_to_cards(self, card1: str, card2: str) -> Optional[str]:
        want = self.get_current_want()

        if want == card1 or want == card2:
            roll = random.random()
            if roll < self.correct_prob:
                return want
            elif roll < self.correct_prob + self.motor_error_prob:
                other = card2 if want == card1 else card1
                return other
            else:
                return None

        if random.random() < 0.05:
            return random.choice([card1, card2])

        return None


CHILD_CLASSES = {
    'consistent': ConsistentChild,
    'noisy': NoisyChild
}


## 9. Tyrinėjimo metodai

In [9]:
class ExplorationStrategy:

    def __init__(self, epsilon: float):
        self.epsilon = epsilon
        self.epsilon_current = epsilon
        self.total_selections = 0
        self.action_counts: Dict[str, int] = defaultdict(int)
        self.last_was_exploration = False

    def reset(self):
        self.epsilon_current = self.epsilon
        self.total_selections = 0
        self.action_counts.clear()

    def select(self, q_values: Dict[str, float], available: List[str]) -> str:
        raise NotImplementedError

    def record_selection(self, action: str):
        self.total_selections += 1
        self.action_counts[action] += 1


class EpsilonGreedy(ExplorationStrategy):

    def select(self, q_values: Dict[str, float], available: List[str]) -> str:
        if random.random() < self.epsilon:
            self.last_was_exploration = True
            return random.choice(available)
        else:
            self.last_was_exploration = False
            best = max(available, key=lambda a: q_values.get(a, 0))
            return best


class DecayingEpsilon(ExplorationStrategy):

    def __init__(self, epsilon: float, decay: float = EPSILON_DECAY, min_epsilon: float = EPSILON_MIN):
        super().__init__(epsilon)
        self.decay = decay
        self.min_epsilon = min_epsilon

    def select(self, q_values: Dict[str, float], available: List[str]) -> str:
        if random.random() < self.epsilon_current:
            self.last_was_exploration = True
            result = random.choice(available)
        else:
            self.last_was_exploration = False
            result = max(available, key=lambda a: q_values.get(a, 0))

        self.epsilon_current = max(self.min_epsilon, self.epsilon_current * self.decay)
        return result


class UCBExploration(ExplorationStrategy):

    def __init__(self, epsilon: float, c: float = UCB_C):
        super().__init__(epsilon)
        self.c = c

    def select(self, q_values: Dict[str, float], available: List[str]) -> str:
        untried = [a for a in available if self.action_counts[a] == 0]
        if untried:
            self.last_was_exploration = True
            return random.choice(untried)

        ucb_values = {}
        for a in available:
            q = q_values.get(a, 0)
            n_a = self.action_counts[a]
            exploration_bonus = self.c * np.sqrt(np.log(self.total_selections + 1) / (n_a + 1))
            ucb_values[a] = q + exploration_bonus

        best = max(available, key=lambda a: ucb_values[a])
        self.last_was_exploration = (ucb_values[best] > q_values.get(best, 0) + 0.1)
        return best


EXPLORATION_CLASSES = {
    'epsilon_greedy': EpsilonGreedy,
    'decaying_epsilon': DecayingEpsilon,
    'ucb': UCBExploration
}

## 10. Bazinis RL modelis ir kortelių parinkimo strategijos

**Strategija A:**
- 1 bandymas: 2 kortelės pagal tyrinėjimo metodą
- 2–3 bandymai: 1 panaši + 1 priešinga atmestoms kortelėms
- 4–5 bandymai: 2 priešingiausios atmestoms kortelėms

**Strategija B:**
- 1 bandymas: 2 kortelės pagal tyrinėjimo metodą
- 2–5 bandymai: 2 panašiausios atmestoms kortelėms

**Strategija C:**
- 1 bandymas: 2 kortelės pagal tyrinėjimo metodą
- 2 bandymas: 1 panaši atmestoms + 1 pagal Q reikšmes
- 3 bandymas: 2 panašiausios atmestoms kortelėms
- 4–5 bandymai: 1 panaši + 1 priešinga atmestoms kortelėms

**Strategija D:**
- 1–5 bandymai: visada 2 kortelės pagal tyrinėjimo metodą / Q reikšmes, atmetimų istorija ignoruojama

**Strategija E:**
- 1 bandymas: 2 kortelės pagal tyrinėjimo metodą
- 2 bandymas: 1 pagal Q reikšmes + 1 panaši atmestoms
- 3 bandymas: sujungiami 3 geriausių pagal Q ir 3 panašiausių sąrašai, parenkamos 2 geriausios pagal Q
- 4–5 bandymai: iš priešingiausių atmestoms parenkama ta, kurios Q reikšmė aukščiausia + 1 pagal Q reikšmes

In [10]:
class BaseRLModel:
    def __init__(self, alpha: float, gamma: float, epsilon: float,
                 exploration_method: str, strategy: str, similarity_method: str):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.exploration_method = exploration_method
        self.strategy = strategy
        self.similarity_method = similarity_method
        self.name = "Base"

        self.q_table: Dict[State, Dict[str, float]] = defaultdict(lambda: defaultdict(float))

        ExplorationClass = EXPLORATION_CLASSES[exploration_method]
        self.exploration = ExplorationClass(epsilon)

        self.current_state: State = State(0, 'start')
        self.session_shown: List[str] = []
        self.last_rejected: List[str] = []

        self.similarity_matrix = SIMILARITY_MATRICES[similarity_method]

    def get_q(self, state: State, food: str) -> float:
        return self.q_table[state][food]

    def get_q_dict(self, state: State) -> Dict[str, float]:
        return dict(self.q_table[state])

    def get_best_q_info(self, state: State, available: List[str]) -> Tuple[str, float]:
        q_vals = [(f, self.get_q(state, f)) for f in available]
        q_vals.sort(key=lambda x: x[1], reverse=True)
        return q_vals[0][0], q_vals[0][1]

    def select_by_exploration(self, state: State, available: List[str]) -> str:
        q_dict = self.get_q_dict(state)
        selected = self.exploration.select(q_dict, available)
        self.exploration.record_selection(selected)
        return selected

    def get_top_cards(self, state: State, available: List[str], n: int = 2) -> List[str]:
        result = []
        remaining = available.copy()
        for _ in range(n):
            if not remaining:
                break
            card = self.select_by_exploration(state, remaining)
            result.append(card)
            remaining.remove(card)
        return result

    def get_similarity_to_rejected(self, card: str) -> float:
        if not self.last_rejected:
            return 0.0
        card_idx = FOOD_LIST.index(card)
        sims = [self.similarity_matrix[card_idx][FOOD_LIST.index(ref)]
                for ref in self.last_rejected if ref in FOOD_LIST]
        return np.mean(sims) if sims else 0.0

    def get_most_similar(self, reference_cards: List[str], available: List[str], n: int = 1) -> List[str]:
        if not reference_cards or not available:
            return available[:n]
        similarities = []
        for food in available:
            food_idx = FOOD_LIST.index(food)
            avg_sim = np.mean([self.similarity_matrix[food_idx][FOOD_LIST.index(ref)]
                              for ref in reference_cards if ref in FOOD_LIST])
            similarities.append((food, avg_sim))
        similarities.sort(key=lambda x: x[1], reverse=True)
        return [s[0] for s in similarities[:n]]

    def get_most_opposite(self, reference_cards: List[str], available: List[str], n: int = 1) -> List[str]:
        if not reference_cards or not available:
            return available[:n]
        similarities = []
        for food in available:
            food_idx = FOOD_LIST.index(food)
            avg_sim = np.mean([self.similarity_matrix[food_idx][FOOD_LIST.index(ref)]
                              for ref in reference_cards if ref in FOOD_LIST])
            similarities.append((food, avg_sim))
        similarities.sort(key=lambda x: x[1])
        return [s[0] for s in similarities[:n]]

    def select_cards(self) -> Tuple[str, str, str]:
        available = [f for f in FOOD_LIST if f not in self.session_shown]
        if len(available) < 2:
            self.session_shown = []
            available = FOOD_LIST.copy()

        rejections = self.current_state.rejections

        if self.strategy == 'A':
            return self._strategy_A(available, rejections)
        elif self.strategy == 'B':
            return self._strategy_B(available, rejections)
        elif self.strategy == 'C':
            return self._strategy_C(available, rejections)
        elif self.strategy == 'D':
            return self._strategy_D(available, rejections)
        else:
            return self._strategy_E(available, rejections)


    def _strategy_A(self, available: List[str], rejections: int) -> Tuple[str, str, str]:
        if rejections == 0:
            cards = self.get_top_cards(self.current_state, available, 2)
            reason = f"TOP2({self.exploration_method})"
        elif rejections <= 2:
            similar = self.get_most_similar(self.last_rejected, available, 1)
            remaining = [f for f in available if f not in similar]
            opposite = self.get_most_opposite(self.last_rejected, remaining, 1) if remaining else similar
            cards = similar + opposite
            reason = f"Panasus+Priesingas(rej={rejections})"
        else:
            # 4-5 bandymai: daugiau priešingų
            cards = self.get_most_opposite(self.last_rejected, available, 2)
            reason = f"2Priesingi(rej={rejections})"
        self.session_shown.extend(cards[:2])
        return (cards[0], cards[1] if len(cards) > 1 else cards[0], reason)

    def _strategy_B(self, available: List[str], rejections: int) -> Tuple[str, str, str]:
        if rejections == 0:
            cards = self.get_top_cards(self.current_state, available, 2)
            reason = f"TOP2({self.exploration_method})"
        else:
            cards = self.get_most_similar(self.last_rejected, available, 2)
            reason = f"2Panasus(rej={rejections})"
        self.session_shown.extend(cards[:2])
        return (cards[0], cards[1] if len(cards) > 1 else cards[0], reason)

    def _strategy_C(self, available: List[str], rejections: int) -> Tuple[str, str, str]:
        if rejections == 0:
            cards = self.get_top_cards(self.current_state, available, 2)
            reason = f"TOP2({self.exploration_method})"
        elif rejections == 1:
            similar = self.get_most_similar(self.last_rejected, available, 1)
            remaining = [f for f in available if f not in similar]
            top_remaining = self.get_top_cards(self.current_state, remaining, 1) if remaining else similar
            cards = similar + top_remaining
            reason = "Panasus+TOP"
        elif rejections == 2:
            cards = self.get_most_similar(self.last_rejected, available, 2)
            reason = "2Panasus"
        else:
            similar = self.get_most_similar(self.last_rejected, available, 1)
            remaining = [f for f in available if f not in similar]
            opposite = self.get_most_opposite(self.last_rejected, remaining, 1) if remaining else similar
            cards = similar + opposite
            reason = f"Mix(rej={rejections})"
        self.session_shown.extend(cards[:2])
        return (cards[0], cards[1] if len(cards) > 1 else cards[0], reason)

    def _strategy_D(self, available: List[str], rejections: int) -> Tuple[str, str, str]:
        cards = self.get_top_cards(self.current_state, available, 2)
        reason = f"LAISVA({self.exploration_method},rej={rejections})"
        self.session_shown.extend(cards[:2])
        return (cards[0], cards[1] if len(cards) > 1 else cards[0], reason)

    def _strategy_E(self, available: List[str], rejections: int) -> Tuple[str, str, str]:
        if rejections == 0:
            cards = self.get_top_cards(self.current_state, available, 2)
            reason = f"KONTEKST_Q(rej=0)"
        elif rejections == 1:
            # 70% Q + 30% panašumas
            q_cards = self.get_top_cards(self.current_state, available, 4)
            sim_cards = self.get_most_similar(self.last_rejected, available, 2)
            card1 = q_cards[0]
            card2 = sim_cards[0] if sim_cards[0] != card1 else (sim_cards[1] if len(sim_cards) > 1 else q_cards[1])
            cards = [card1, card2]
            reason = f"KONTEKST_MIX(rej=1)"
        elif rejections == 2:
            # 50/50
            q_cards = self.get_top_cards(self.current_state, available, 3)
            sim_cards = self.get_most_similar(self.last_rejected, available, 3)
            combined = list(set(q_cards + sim_cards))
            combined_q = [(c, self.get_q(self.current_state, c)) for c in combined]
            combined_q.sort(key=lambda x: x[1], reverse=True)
            cards = [combined_q[0][0], combined_q[1][0]] if len(combined_q) > 1 else [combined_q[0][0], combined_q[0][0]]
            reason = f"KONTEKST_DEEP(rej=2)"
        else:
            # 4-5: daugiau priešingų + Q
            opposite = self.get_most_opposite(self.last_rejected, available, 3)
            best_opposite = max(opposite, key=lambda c: self.get_q(self.current_state, c))
            remaining = [f for f in available if f != best_opposite]
            second = self.get_top_cards(self.current_state, remaining, 1)[0] if remaining else best_opposite
            cards = [best_opposite, second]
            reason = f"KONTEKST_EXTREME(rej={rejections})"

        self.session_shown.extend(cards[:2])
        return (cards[0], cards[1] if len(cards) > 1 else cards[0], reason)

    def start_new_session(self):
        self.session_shown = []
        self.last_rejected = []
        self.current_state = State(0, 'start')

    def record_rejection(self, cards: List[str]):
        self.last_rejected = cards.copy()

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        raise NotImplementedError

## 11. Skatinamojo mokymosi algoritmai

In [11]:
class QLearningModel(BaseRLModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.name = "Q-Learning"

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        card1, card2 = action
        max_next_q = max(self.get_q(next_state, f) for f in FOOD_LIST)

        if chosen is None:
            for card in [card1, card2]:
                old_q = self.get_q(state, card)
                td_target = reward + self.gamma * max_next_q
                self.q_table[state][card] = old_q + self.alpha * (td_target - old_q)
        else:
            other = card2 if chosen == card1 else card1

            old_q = self.get_q(state, chosen)
            td_target = reward + self.gamma * max_next_q
            self.q_table[state][chosen] = old_q + self.alpha * (td_target - old_q)

            old_q_other = self.get_q(state, other)
            td_target_other = REWARDS['other_card'] + self.gamma * max_next_q
            self.q_table[state][other] = old_q_other + self.alpha * (td_target_other - old_q_other)

        return (self.get_q(state, card1), self.get_q(state, card2))


class SARSAModel(BaseRLModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.name = "SARSA"

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        card1, card2 = action
        available = [f for f in FOOD_LIST if f not in self.session_shown]
        if len(available) >= 2:
            next_card = self.select_by_exploration(next_state, available)
            next_q = self.get_q(next_state, next_card)
        else:
            next_q = 0

        if chosen is None:
            for card in [card1, card2]:
                old_q = self.get_q(state, card)
                td_target = reward + self.gamma * next_q
                self.q_table[state][card] = old_q + self.alpha * (td_target - old_q)
        else:
            other = card2 if chosen == card1 else card1

            old_q = self.get_q(state, chosen)
            td_target = reward + self.gamma * next_q
            self.q_table[state][chosen] = old_q + self.alpha * (td_target - old_q)

            old_q_other = self.get_q(state, other)
            td_target_other = REWARDS['other_card'] + self.gamma * next_q
            self.q_table[state][other] = old_q_other + self.alpha * (td_target_other - old_q_other)

        return (self.get_q(state, card1), self.get_q(state, card2))


class ExpectedSARSAModel(BaseRLModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.name = "Expected-SARSA"

    def get_expected_q(self, state: State, available: List[str]) -> float:
        if not available:
            return 0.0
        q_vals = [self.get_q(state, f) for f in available]
        max_q = max(q_vals)
        mean_q = sum(q_vals) / len(q_vals)
        return (1 - self.epsilon) * max_q + self.epsilon * mean_q

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        card1, card2 = action
        available = [f for f in FOOD_LIST if f not in self.session_shown]
        if len(available) < 2:
            available = FOOD_LIST.copy()
        expected_q = self.get_expected_q(next_state, available)

        if chosen is None:
            for card in [card1, card2]:
                old_q = self.get_q(state, card)
                td_target = reward + self.gamma * expected_q
                self.q_table[state][card] = old_q + self.alpha * (td_target - old_q)
        else:
            other = card2 if chosen == card1 else card1

            old_q = self.get_q(state, chosen)
            td_target = reward + self.gamma * expected_q
            self.q_table[state][chosen] = old_q + self.alpha * (td_target - old_q)

            old_q_other = self.get_q(state, other)
            td_target_other = REWARDS['other_card'] + self.gamma * expected_q
            self.q_table[state][other] = old_q_other + self.alpha * (td_target_other - old_q_other)

        return (self.get_q(state, card1), self.get_q(state, card2))


class DoubleQLearningModel(BaseRLModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.name = "Double-Q-Learning"
        self.q_table_b: Dict[State, Dict[str, float]] = defaultdict(lambda: defaultdict(float))

    def get_q_b(self, state: State, food: str) -> float:
        return self.q_table_b[state][food]

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        card1, card2 = action

        if random.random() < 0.5:
            best_action = max(FOOD_LIST, key=lambda f: self.get_q(next_state, f))
            max_next_q = self.get_q_b(next_state, best_action)
            q_table = self.q_table
        else:
            best_action = max(FOOD_LIST, key=lambda f: self.get_q_b(next_state, f))
            max_next_q = self.get_q(next_state, best_action)
            q_table = self.q_table_b

        if chosen is None:
            for card in [card1, card2]:
                old_q = q_table[state][card]
                td_target = reward + self.gamma * max_next_q
                q_table[state][card] = old_q + self.alpha * (td_target - old_q)
        else:
            other = card2 if chosen == card1 else card1

            old_q = q_table[state][chosen]
            td_target = reward + self.gamma * max_next_q
            q_table[state][chosen] = old_q + self.alpha * (td_target - old_q)

            old_q_other = q_table[state][other]
            td_target_other = REWARDS['other_card'] + self.gamma * max_next_q
            q_table[state][other] = old_q_other + self.alpha * (td_target_other - old_q_other)

        return ((self.get_q(state, card1) + self.get_q_b(state, card1)) / 2,
                (self.get_q(state, card2) + self.get_q_b(state, card2)) / 2)


class ActorCriticModel(BaseRLModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.name = "Actor-Critic"
        self.policy: Dict[State, Dict[str, float]] = defaultdict(lambda: defaultdict(lambda: 1.0 / len(FOOD_LIST)))
        self.value_table: Dict[State, float] = defaultdict(float)

    def get_policy_prob(self, state: State, food: str) -> float:
        return self.policy[state][food]

    def get_q_dict(self, state: State) -> Dict[str, float]:
        return {f: self.get_policy_prob(state, f) for f in FOOD_LIST}

    def update(self, state: State, action: Tuple[str, str], reward: float,
               next_state: State, chosen: Optional[str] = None) -> Tuple[float, float]:
        card1, card2 = action

        v_current = self.value_table[state]
        v_next = self.value_table[next_state]
        td_error = reward + self.gamma * v_next - v_current

        self.value_table[state] = v_current + self.alpha * td_error

        if chosen is not None:
            self.policy[state][chosen] += self.alpha * td_error
            other = card2 if chosen == card1 else card1
            self.policy[state][other] -= self.alpha * 0.1 * abs(td_error)
        else:
            for card in [card1, card2]:
                self.policy[state][card] += self.alpha * td_error

        return (self.get_policy_prob(state, card1), self.get_policy_prob(state, card2))


MODEL_CLASSES = {
    'Q-Learning': QLearningModel,
    'SARSA': SARSAModel,
    'Expected-SARSA': ExpectedSARSAModel,
    'Double-Q-Learning': DoubleQLearningModel,
    'Actor-Critic': ActorCriticModel
}

## 12. Simuliacijos funkcija

In [12]:
def run_simulation(model: BaseRLModel, child: VirtualChild,
                   logger: ResultsLogger, max_attempts: int = MAX_ATTEMPTS) -> Dict:

    child.reset()
    model.start_new_session()
    model.exploration.reset()
    logger.reset()

    session_num = 0
    sessions_succeeded = 0
    total_attempts = 0

    while child.get_current_want() is not None:
        session_num += 1
        wanted = child.get_current_want()
        model.start_new_session()

        session_success = False
        session_attempts = 0

        for attempt_num in range(1, max_attempts + 1):
            session_attempts += 1
            state = model.current_state
            card1, card2, reason = model.select_cards()

            chosen = child.react_to_cards(card1, card2)

            if chosen is not None:
                reward = REWARDS['success']

                next_state = State(0, 'success')
                model.update(state, (card1, card2), reward, next_state, chosen)
                model.current_state = next_state

                session_success = True
                sessions_succeeded += 1

                logger.log_attempt(
                    session=session_num, attempt=attempt_num, wanted=wanted,
                    card1=card1, card2=card2, result='SUCCESS', chosen=chosen,
                    model=model.name, strategy=model.strategy,
                    exploration=model.exploration_method
                )
                break
            else:
                next_state = State(min(state.rejections + 1, max_attempts - 1), 'rejection')
                model.update(state, (card1, card2), REWARDS['rejection'], next_state, None)
                model.record_rejection([card1, card2])
                model.current_state = next_state

                logger.log_attempt(
                    session=session_num, attempt=attempt_num, wanted=wanted,
                    card1=card1, card2=card2, result='REJECTION', chosen='',
                    model=model.name, strategy=model.strategy,
                    exploration=model.exploration_method
                )

        total_attempts += session_attempts
        logger.log_session(session_success, session_attempts)
        child.next_want()

    return {
        'model': model.name,
        'exploration': model.exploration_method,
        'strategy': model.strategy,
        'alpha': model.alpha,
        'gamma': model.gamma,
        'epsilon': model.epsilon,
        'similarity': model.similarity_method,
        'child_profile': child.profile_name,
        'total_sessions': session_num,
        'sessions_succeeded': sessions_succeeded,
        'success_rate': logger.get_success_rate(),
        'avg_attempts': logger.get_avg_attempts(),
        'max_success_streak': logger.max_success_streak,
    }

## 13. Eksperimentų paleidimas

In [13]:
def run_experiment(child_profile: str, n_runs: int = N_RUNS) -> pd.DataFrame:

    print(f"\n{'-'*40}")
    print(f"{child_profile.upper()} virtualus vaikas")
    print(f"\n")

    all_combinations = list(itertools.product(
        MODEL_CLASSES.keys(),
        PARAM_GRID['exploration_method'],
        PARAM_GRID['strategy'],
        PARAM_GRID['alpha'],
        PARAM_GRID['gamma'],
        PARAM_GRID['epsilon'],
        PARAM_GRID['similarity_method']
    ))

    print(f"Konfigūracijų: {len(all_combinations)}")
    print(f"Pakartojimų: {n_runs}")
    print(f"Iš viso simuliacijų: {len(all_combinations) * n_runs}")

    results = []
    ChildClass = CHILD_CLASSES[child_profile]

    for i, (model_name, expl, strat, alpha, gamma, eps, sim) in enumerate(all_combinations):
        run_results = []

        for run in range(n_runs):

            ModelClass = MODEL_CLASSES[model_name]
            model = ModelClass(
                alpha=alpha, gamma=gamma, epsilon=eps,
                exploration_method=expl, strategy=strat,
                similarity_method=sim
            )

            child = ChildClass(CHILD_WANTS, SIMILARITY_MATRICES[sim])
            logger = ResultsLogger()

            result = run_simulation(model, child, logger)
            run_results.append(result)

        avg_result = {
            'model': model_name,
            'exploration': expl,
            'strategy': strat,
            'alpha': alpha,
            'gamma': gamma,
            'epsilon': eps,
            'similarity': sim,
            'child_profile': child_profile,
            'success_rate_mean': np.mean([r['success_rate'] for r in run_results]),
            'success_rate_std': np.std([r['success_rate'] for r in run_results]),
            'avg_attempts_mean': np.mean([r['avg_attempts'] for r in run_results]),
            'avg_attempts_std': np.std([r['avg_attempts'] for r in run_results]),
        }
        results.append(avg_result)

        if (i + 1) % 100 == 0:
            print(f"  [{i+1}/{len(all_combinations)}] Baigta...")

    print(f"Eksperimentas baigtas: {len(results)} konfigūracijos")
    return pd.DataFrame(results)

In [ ]:

all_results = {}

for profile in CHILD_PROFILES:
    all_results[profile] = run_experiment(profile, n_runs=N_RUNS)

print("\n")
print("VISI EKSPERIMENTAI BAIGTI")

## 14. Rezultatai

In [ ]:
def analyze_results(results_df: pd.DataFrame, profile_name: str):
    print("\n" + "-"*40)
    print(f"{profile_name.upper()} virutalus vaikas")
    print(f"\n")

    print("\nTOP 10 konfigūracijų:")
    top10 = results_df.nlargest(10, 'success_rate_mean')[[
        'model', 'strategy', 'exploration', 'success_rate_mean', 'success_rate_std', 'avg_attempts_mean'
    ]]
    print(top10.to_string(index=False))

    print(f"\n")
    print("VIDURKIAI PAGAL PARAMETRUS:")

    for param in ['model', 'strategy', 'exploration']:
        print(f"\n{param.upper()}:")
        grouped = results_df.groupby(param).agg({
            'success_rate_mean': ['mean', 'std'],
            'avg_attempts_mean': 'mean'
        }).round(2)
        grouped.columns = ['Vidurkis %', 'Std %', 'Bandymų']
        grouped = grouped.sort_values('Vidurkis %', ascending=False)
        for idx, row in grouped.iterrows():
            print(f"  {idx}: {row['Vidurkis %']:.1f}% (±{row['Std %']:.1f}%), bandymų: {row['Bandymų']:.2f}")

for profile, df in all_results.items():
    analyze_results(df, profile)

## 15. Eksportas

In [16]:
for profile, df in all_results.items():
    filename = f'rezultatai_{profile}.csv'
    df.to_csv(filename, index=False)
    print(f"Išsaugota: {filename}")

summary_rows = []
for profile, df in all_results.items():
    for model in MODEL_NAMES:
        model_df = df[df['model'] == model]
        summary_rows.append({
            'profile': profile,
            'model': model,
            'success_rate': model_df['success_rate_mean'].mean(),
            'avg_attempts': model_df['avg_attempts_mean'].mean()
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('bendra_suvestine.csv', index=False)
print("Išsaugota: bendra_suvestine.csv")

Išsaugota: rezultatai_consistent.csv
Išsaugota: rezultatai_noisy.csv
Išsaugota: bendra_suvestine.csv


## 16. Išvados

In [ ]:
for profile, df in all_results.items():
    print(f"\n{profile.upper()} vaikas:")

    best_model = df.groupby('model')['success_rate_mean'].mean().idxmax()
    best_model_score = df.groupby('model')['success_rate_mean'].mean().max()
    print(f"  Geriausias algoritmas: {best_model} ({best_model_score:.1f}%)")

    best_strat = df.groupby('strategy')['success_rate_mean'].mean().idxmax()
    best_strat_score = df.groupby('strategy')['success_rate_mean'].mean().max()
    print(f"  Geriausia strategija: {best_strat} ({best_strat_score:.1f}%)")

    best_expl = df.groupby('exploration')['success_rate_mean'].mean().idxmax()
    best_expl_score = df.groupby('exploration')['success_rate_mean'].mean().max()
    print(f"  Geriausias tyrinėjimas: {best_expl} ({best_expl_score:.1f}%)")